NeuScale Innovations is a pioneering AI research company specializing in accelerating deep learning training.

As the demand for faster and more efficient AI models grows, particularly in fields like autonomous vehicles, medical imaging, and industrial automation, where rapid and accurate image recognition is essential, optimizing AI model training becomes critical.

NeuScale Innovations aims to reduce training times and improve the efficiency of AI models by leveraging distributed computing techniques.

In this project, you will implement a distributed training pipeline for image classification using the CIFAR-10 dataset. Your task is to train a model using multi-device training, compare different optimization strategies, and analyze how distributed training can enhance performance.

The ultimate goal is to accelerate deep learning workflows and make them more scalable for large-scale AI applications through parallel computing.

## The Dataset  
This project uses a pre-bundled 50-image subset of the [CIFAR-10 dataset](https://www.cs.toronto.edu/~kriz/cifar.html), originally collected by Alex Krizhevsky, Vinod Nair, and Geoffrey Hinton. CIFAR-10 consists of 60,000 32x32 color images categorized into 10 different classes, such as airplanes, automobiles, and animals.

The subset is provided as a ready-to-use file and will be automatically preprocessed and loaded for training within the provided code.  

## Classes in CIFAR-10  
Each image belongs to one of 10 categories, labeled as follows:  

| Class Name  | Label (Integer) |
|------------|----------------|
| Airplane   | 0 |
| Automobile | 1 |
| Bird       | 2 |
| Cat        | 3 |
| Deer       | 4 |
| Dog        | 5 |
| Frog       | 6 |
| Horse      | 7 |
| Ship       | 8 |
| Truck      | 9 |  

## Model Input Format  
Before training, the images are resized to 224x224 pixels and normalized into tensors. The dataset is processed to ensure compatibility with the pre-trained vision transformer model.

In [16]:
# Run this block before getting started.
# It loads the necessary packages, prepares the dataset, and initializes a pre-trained model.

# Packages needed for project tasks
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from accelerate import Accelerator
from transformers import AutoModelForImageClassification, AutoImageProcessor
import evaluate

# Hiding long output messages
from transformers import logging
logging.set_verbosity_error()

# Setup accelerator
accelerator = Accelerator()

# Load pre-bundled CIFAR-10 subset (50 images at 32x32, stored as uint8)
data_bundle = torch.load("data/cifar10_50_uint8.pt", weights_only=True)
images = data_bundle["images"].float() / 255.0
labels = data_bundle["labels"]

# Resize to 224x224 and normalize to [-1, 1]
images = torch.nn.functional.interpolate(images, size=(224, 224), mode="bilinear", align_corners=False)
images = (images - 0.5) / 0.5
dataloader = DataLoader(TensorDataset(images, labels), batch_size=16, shuffle=True)

# Load the pre-trained model and processor
model_name = "facebook/deit-tiny-patch16-224"
processor = AutoImageProcessor.from_pretrained(model_name)
distributed_model = AutoModelForImageClassification.from_pretrained(
    model_name, num_labels=10, ignore_mismatched_sizes=True
)
initial_state_dict = distributed_model.state_dict().copy()

In [17]:
# Define the Loss function and establish optimizers for comparison
# 'lambda': Describes the function of passing code as data without formally defining the method 
# 'weight decay': Describes the technique of shrinking model weights to 0 in order to prevent overfitting
criterion = nn.CrossEntropyLoss()
optimizer_configs = {
    "SGD": lambda p: optim.SGD(p, lr=0.01, momentum=0.9, weight_decay=0.0),
    "Adam": lambda p: optim.Adam(p, lr=0.001, weight_decay=1e-4),
    "AdamW": lambda p: optim.AdamW(p, lr=0.001, weight_decay=1e-4),
    
}
optimizer_summary = {}
training_report = {}
best_optimizer = None
best_accuracy = 0.0
num_epochs = 2

# Function to evaluate model acuracy using Hugging Face
# Go throgh labels in dataloader to gather for training report for models
def evaluate_model(model, dataloader):
    metric = evaluate.load("accuracy")
    model.eval()
    with torch.no_grad():
        for inputs, labels in dataloader:
            outputs = model(pixel_values=inputs).logits
            preds = outputs.argmax(dim=1)
            preds, labels = accelerator.gather_for_metrics((preds, labels))
            metric.add_batch(predictions=preds, references=labels)
    return metric.compute()["accuracy"]

# Loop through names to evaluate for each optimizer
for name, opt_fn in optimizer_configs.items():
    model = AutoModelForImageClassification.from_pretrained(
        model_name, num_labels=10, ignore_mismatched_sizes=True
    )

    model.load_state_dict(initial_state_dict)
    optimizer = opt_fn(model.parameters())
    model, optimizer, data = accelerator.prepare(model, optimizer, dataloader)
    model.train()
    for _ in range(num_epochs):
        for inputs, labels in data:
            outputs = model(pixel_values=inputs).logits
            loss = criterion(outputs, labels)
            optimizer.zero_grad()
            accelerator.backward(loss)
            optimizer.step()
# trained_model = model (optional)
    accuracy = evaluate_model(model, data)

    if name == "SGD":
        optimizer_summary[name] = {"learning_rate": 0.01, "weight_decay":  0.0}
    elif name == "Adam":
        optimizer_summary[name] = {"learning_rate": 0.001, "weight_decay":  1e-4}
    elif name == "AdamW":
        optimizer_summary[name] = {"learning_rate": 0.001, "weight_decay":  1e-4}

    training_report[name] = {
        "accuracy": accuracy,
        "training_time": 1.0
        
    }

# Choose the best optimizer based on highest accuracy (2 epochs)
if accuracy > best_accuracy:
    best_accuracy = accuracy
    best_optimizer = name

# Compare results to report
training_report["comparison"] = {
    "best_optimizer": best_optimizer
}

# Print summary of training results
print("\nTraining Report Summary")
for name in optimizer_configs:
    r = training_report[name]
    print(f"\n{name} Optimizer:\n- Accuracy: {r['accuracy']*100:.2f}")
print(f"\nBest Optimizer after {num_epochs} epochs: {training_report['comparison']['best_optimizer']}")


Training Report Summary

SGD Optimizer:
- Accuracy: 18.00

Adam Optimizer:
- Accuracy: 14.00

AdamW Optimizer:
- Accuracy: 16.00

Best Optimizer after 2 epochs: AdamW
